## Setup

In [2]:
import polars as pl
import matplotlib.pyplot as plt
import requests

import warnings
warnings.filterwarnings("ignore")
import os
from pathlib import Path
import sys
project_root = Path().resolve().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added {project_root.stem} to sys.path")

from src.utils.llama import DefiLlamaAPI

llama = DefiLlamaAPI()


Added stables to sys.path


## total supply

In [3]:
df_total_supply = llama.get_total_supply()
# df_total_supply = df_total_supply.drop(['circulatingPrevWeek', 'circulatingPrevMonth'])#.sort('change_24h_pct', descending=True)


In [4]:
df_total_supply

id,name,symbol,gecko_id,pegType,priceSource,pegMechanism,circulating,circulatingPrevDay,change_24h,change_24h_pct
str,str,str,str,str,str,str,f64,f64,f64,f64
"""1""","""Tether""","""USDT""","""tether""","""peggedUSD""","""defillama""","""fiat-backed""",1.2032e11,1.2032e11,4.2496e6,0.0
"""2""","""USD Coin""","""USDC""","""usd-coin""","""peggedUSD""","""defillama""","""fiat-backed""",3.4413e10,3.4353e10,6.0289e7,0.18
"""5""","""Dai""","""DAI""","""dai""","""peggedUSD""","""defillama""","""crypto-backed""",4.9048e9,4.9310e9,-2.6247e7,-0.53
"""146""","""Ethena USDe""","""USDe""","""ethena-usde""","""peggedUSD""","""coingecko""","""crypto-backed""",2.6216e9,2.6215e9,160197.708689,0.01
"""119""","""First Digital USD""","""FDUSD""","""first-digital-usd""","""peggedUSD""","""defillama""","""fiat-backed""",2.2777e9,2.3061e9,-2.8397e7,-1.23
…,…,…,…,…,…,…,…,…,…,…
"""182""","""Zunami ETH""","""zunETH""","""zunETH""","""peggedVAR""","""defillama""","""crypto-backed""",null,null,null,null
"""183""","""Bitcoin USD""","""BtcUSD""","""bitcoin-usd-btcfi""","""peggedVAR""","""defillama""","""crypto-backed""",null,null,null,null
"""186""","""International Stable Currency""","""ISC""","""international-stable-currency""","""peggedVAR""","""coingecko""","""fiat-backed""",null,null,null,null


In [5]:
df_total_supply['pegMechanism'].value_counts()

pegMechanism,count
str,u32
"""fiat-backed""",50
"""algorithmic""",27
"""crypto-backed""",127


## big movers

In [24]:
import plotly.express as px

fig = px.scatter(
    df_total_supply,
    x='change_24h_pct',
    y='change_24h',
    opacity=0.5,
    color_discrete_sequence=['blue'],
    labels={
        'change_24h_pct': 'a',
        'change_24h': '24h Change'
    },
    title='Stablecoin 24h Changes'
)
fig.show()



In [29]:
ll = df_total_supply.filter(
    (abs(pl.col('change_24h_pct')) > 5) 
    & (pl.col('circulating') > 1e6)
    & (pl.col('pegMechanism') != "fiat-backed")
)['name'].to_list()


In [30]:
ll

['sUSD', 'USDT+', 'Zunami USD']

In [9]:
df_chains = llama.get_chain_circulation()


In [31]:
df_chains_filtered = df_chains.filter(
    pl.col('coin').is_in(ll)
    & (pl.col('change_24h') != 0.0)
)#.sort('change_24h_pct', descending=True)


In [34]:
df_chains_filtered#.group_by('coin').agg(pl.col('change_24h').sum())

coin,symbol,chain,current,prev_day,price,change_24h,change_24h_pct
str,str,str,f64,f64,f64,f64,f64
"""sUSD""","""SUSD""","""Optimism""",1.9726e7,2.2301e7,1.001,-2.5743e6,-11.54
"""sUSD""","""SUSD""","""Ethereum""",1.0912e7,1.1484e7,1.001,-572031.655245,-4.98
"""USDT+""","""USDT+""","""Arbitrum""",2.9380e6,3.1115e6,0.997656,-173474.29628,-5.58
"""USDT+""","""USDT+""","""Linea""",18368.464737,18385.186909,0.997656,-16.722172,-0.09
"""Zunami USD""","""zunUSD""","""Ethereum""",1.9436e6,2.1640e6,0.992873,-220325.86824,-10.18


In [35]:
df_total_supply.filter(
    pl.col('name').is_in(ll)
    # & (pl.col('change_24h') != 0.0)
)#.sort('change_24h_pct', descending=True)


id,name,symbol,gecko_id,pegType,priceSource,pegMechanism,circulating,circulatingPrevDay,change_24h,change_24h_pct
str,str,str,str,str,str,str,f64,f64,f64,f64
"""22""","""sUSD""","""SUSD""","""nusd""","""peggedUSD""","""defillama""","""crypto-backed""",3.0639e7,3.3785e7,-3.1463e6,-9.31
"""112""","""USDT+""","""USDT+""","""usdtplus""","""peggedUSD""","""defillama""","""crypto-backed""",2.9851e6,3.1586e6,-173491.018452,-5.49
"""181""","""Zunami USD""","""zunUSD""","""zunUSD""","""peggedUSD""","""defillama""","""crypto-backed""",1.9436e6,2.1640e6,-220325.86824,-10.18


In [ ]:
name = 'sUSD'
chain = "Optimism"
contract_address = "0x8c6f28f2f1a3c87f0f938b96d27520d9751ec8d9"
"""
Transfer
43422
Approval
8446
Issued
2125
Burned
444
"""




In [59]:
# Read first 5 lines as column names
with open('c.txt', 'r') as f:
    columns = [next(f).strip() for _ in range(5)]

# Read remaining lines in groups of 5 as data rows
data = []
with open('c.txt', 'r') as f:
    # Skip the column headers
    for _ in range(5):
        next(f)
        
    # Read data rows
    while True:
        try:
            row = [next(f).strip() for _ in range(5)]
            data.append(row)
        except StopIteration:
            break

# Create DataFrame
df_transfers = pl.DataFrame(
    data,
    schema=columns
)
df_transfers = df_transfers.with_columns(
    pl.col('total_amount').cast(pl.Float64)
)
df_transfers


transfer_from,transfer_to,tx_from,tx_to,total_amount
str,str,str,str,f64
"""0xe2101d19b78e695982975cc1a27b…","""0xbce24f7e2bfd63e590bf18524e80…","""0x2e8d0cc9ab070d1336188c907b56…","""0xb8b2ec7346c3989e13cf03f22885…",2.9247e7
"""0xbce24f7e2bfd63e590bf18524e80…","""0xe2101d19b78e695982975cc1a27b…","""0x2e8d0cc9ab070d1336188c907b56…","""0xb8b2ec7346c3989e13cf03f22885…",2.8285e7
"""0xcc7d6ed524760539311ed0cdb41d…","""0x0000000000000000000000000000…","""0xc8c3ba0fc3f934a766ba51efc742…","""0xcc7d6ed524760539311ed0cdb41d…",1.3956e6
"""0x7d3c9c6566375d7ad6e89169ca5c…","""0x0000000000000000000000000000…","""0x7ce3c65e832ca68c8f364ce294e8…","""0x7d3c9c6566375d7ad6e89169ca5c…",1.2288e6
"""0xe2101d19b78e695982975cc1a27b…","""0xbce24f7e2bfd63e590bf18524e80…","""0xc2d9f9840bfc6a0f48f5dfef6868…","""0xe82343a116d2179f197111d92f9b…",1.0939e6
…,…,…,…,…
"""0xd712e1bf0090691cfba1f74ce2d9…","""0x6d80113e533a2c0fe82eabd35f18…","""0xd712e1bf0090691cfba1f74ce2d9…","""0x794a61358d6845594f94dc1db02a…",165567.424016
"""0x88b3dc3dcd06f1489d1879f4e465…","""0x0000000000000000000000000000…","""0x11e82bbd9477f62562c88b140d2d…","""0x88b3dc3dcd06f1489d1879f4e465…",159320.0
"""0x3bdd90f6b6c6eb350c09d0dfe55a…","""0x0000000000000000000000000000…","""0x0514f2f3e0277c47117e3f33d939…","""0x3bdd90f6b6c6eb350c09d0dfe55a…",158000.0


In [65]:
labels = {
    '0xe2101d19b78e695982975cc1a27b6a9117e02b09': 'UniswapV3Pool',
    '0xbce24f7e2bfd63e590bf18524e80b222f7c982d8': 'beefy_BeaconProxy',
    '0x2e8d0cc9ab070d1336188c907b565565f9c03bb5': 'chad_1',
    '0xb8b2ec7346c3989e13cf03f228851237cd9a9c26': 'chad_contract',
    '0xcc7d6ed524760539311ed0cdb41d0852b4eb77eb': 'solana_bull_3x',
    '0x7d3c9c6566375d7ad6e89169ca5c01b5edc15364': 'solana_bull_2x',
    '0xc8c3ba0fc3f934a766ba51efc742ca359dae82bd': 'chad_2',
    '0xe82343a116d2179f197111d92f9b53611b43c01c': 'BeefyZapRouter',
    '0xfb73dbc7ead4671edac2f471ad50e205cf352c68': 'chad_3',
    '0x7ce3c65e832ca68c8f364ce294e80b603376acff': 'chad_4',
    '0x0000000000000000000000000000000000000000': 'zero_address',
    '0xbc5a22f6df95f987ceef999cc812ed272a9b938a': 'chad_5',
    '0xc2d9f9840bfc6a0f48f5dfef686878534fca5b22': 'chad_6',


}

In [66]:

df_transfers.with_columns(
    pl.col('transfer_from').replace(labels).alias('transfer_from'),
    pl.col('transfer_to').replace(labels).alias('transfer_to'),
    pl.col('tx_from').replace(labels).alias('tx_from'),
    pl.col('tx_to').replace(labels).alias('tx_to')

).head(10)
# 2024-10-28 22:31:04, last 24h

transfer_from,transfer_to,tx_from,tx_to,total_amount
str,str,str,str,f64
"""UniswapV3Pool""","""beefy_BeaconProxy""","""chad_1""","""chad_contract""",2.9247e7
"""beefy_BeaconProxy""","""UniswapV3Pool""","""chad_1""","""chad_contract""",2.8285e7
"""solana_bull_3x""","""zero_address""","""chad_2""","""solana_bull_3x""",1.3956e6
"""solana_bull_2x""","""zero_address""","""chad_4""","""solana_bull_2x""",1.2288e6
"""UniswapV3Pool""","""beefy_BeaconProxy""","""chad_6""","""BeefyZapRouter""",1.0939e6
"""beefy_BeaconProxy""","""UniswapV3Pool""","""chad_6""","""BeefyZapRouter""",1.0911e6
"""UniswapV3Pool""","""beefy_BeaconProxy""","""0x59e7b9be88b69ec43dc0ae9da93a…","""BeefyZapRouter""",1.0833e6
"""beefy_BeaconProxy""","""UniswapV3Pool""","""0x59e7b9be88b69ec43dc0ae9da93a…","""BeefyZapRouter""",1.0751e6
"""beefy_BeaconProxy""","""UniswapV3Pool""","""0x417b4adc279743fc49f047c323fc…","""0x01051113d81d7d6da508462f2ad6…",1.0485e6
